# Shape Analysis | Project 1 - Primitive Fitting

In this project, you will implement a shape abstraction method that fits a set of simple geometric primitives to the surface of an input mesh.\
To do this, you will first implement a local feature descriptor whose features can then be used to extract regions (clusters of faces) that belong to the same primitive.\
Finally, you will fit a parametric cylinder to each cluster of faces and visualize the resulting set of cylinders.

There is no single solution to this problem. You are encouraged to experiment with design choices and parameters. However, you should aim for a meaningful abstraction of the given meshes, and we expect you to justify your design choices in the presentation.\
We provide three example meshes to test your implementation. You can find them in the data folder.

**Some important notes:**

- Make sure your final submission notebook can correctly process and visualize all three meshes, as well as the intermediate results (shape diameter function, regions before and after merging, ...).
- Which mesh gets processed and visualized is controlled by the `MODEL` variable at the top of the notebook. Also define your hyperparameters in the corresponding match section.\
  We will use this variable to test your implementation on the three meshes for grading.
- The notebook should run without errors and terminate in a reasonable amount of time.
- **If you have any questions, come to the tutorials!**

# Example: Initial Regions, After Merging, Cylinders
![image.png](image.png)

## Setup

The most important libraries for this project are numpy, trimesh, and k3d. Read the documentation for these libraries to understand how to use them. You can install them using pip.
You are also allowed to use scipy.

**Important: Do not use any other external libraries for solving the project. You are allowed to use the default libraries of Python. For visualization, please use k3d.**

Below is the match statement to help you manage the parameters for different meshes.

*Hint: You may need to normalize and preprocess your mesh before you can effectively apply the cylinder fitting. Inspect your intermediate results (e.g. from region growing) with k3d.*

In [395]:
import numpy as np
import trimesh as tm
import k3d
from pathlib import Path
from k3d.colormaps import matplotlib_color_maps as cmaps

In [396]:
def rotation_matrix_from_vectors(vec1, vec2):
    ''' Compute the rotation matrix to rotate vec1 to vec2. '''

    a = vec1 / np.linalg.norm(vec1)
    b = vec2 / np.linalg.norm(vec2)

    v = np.cross(a, b)
    c = np.dot(a, b)
    s = np.linalg.norm(v)

    K = np.array([
        [0, -v[2], v[1]],
        [v[2], 0, -v[0]],
        [-v[1], v[0], 0],
    ])

    R = np.eye(3) + K + K @ K * (1 - c) / (s ** 2)

    return R


def get_cylinder_mesh(centroid, axis, height, radius):
    ''' The centroid lies at the center of the cylinder. Therefore, it extends by height/2 in both directions. '''

    cylinder = tm.creation.cylinder(radius=radius, height=height, sections=16)

    R = rotation_matrix_from_vectors(np.array([0, 0, 1]), axis)
    cylinder.vertices = cylinder.vertices @ R.T
    cylinder.vertices += centroid

    return cylinder

In [ ]:
MODEL = 'spot'

DATA_DIR = Path("data")

match MODEL:
    case 'spot':
        FILEPATH = DATA_DIR / 'spot.obj'
        NUM_RAYS = 40
        CONE_ANGLE = 60.0
        SDF_ALPHA = 4.0
        SIMILARITY_THRESHOLD = 0.14
        CONVEX_RATIO = 0.95
        MERGE_MIN_SIZE = 20
        MERGE_FEATURE_THRESHOLD = 0.1


    case 'lion':
        FILEPATH = DATA_DIR / 'lion.obj'
        NUM_RAYS = 30
        CONE_ANGLE = 60.0
        SDF_ALPHA = 4.0
        SIMILARITY_THRESHOLD = 0.02
        CONVEX_RATIO = 1.05
        MERGE_MIN_SIZE = 10

    case 'penguin':
        FILEPATH = DATA_DIR / 'penguin.obj'
        NUM_RAYS = 30
        CONE_ANGLE = 60.0
        SDF_ALPHA = 4.0
        SIMILARITY_THRESHOLD = 0.05
        CONVEX_RATIO = 1.3
        MERGE_MIN_SIZE = 30

    case _:
        raise ValueError(f"Unknown model: {MODEL}")

In [398]:
def load_and_preprocess_mesh(filepath):
    filepath = Path(filepath)

    if not filepath.exists():
        raise FileNotFoundError(f"Mesh file not found: {filepath}")

    mesh = tm.load(str(filepath), force='mesh')

    # rebuild without texture/uv data
    mesh = tm.Trimesh(
        vertices=np.asarray(mesh.vertices),
        faces=np.asarray(mesh.faces),
        process=True
    )

    mesh.fix_normals()

    # center and scale to unit size
    mesh.apply_translation(-mesh.bounding_box.centroid)
    mesh.apply_scale(1.0 / mesh.scale)

    match MODEL:
        case 'spot':
            rot_z = tm.transformations.rotation_matrix(np.radians(180), [0, 0, 1])
            rot_x = tm.transformations.rotation_matrix(np.radians(-90), [1, 0, 0])
            mesh.apply_transform(rot_x @ rot_z)
        case 'lion' | 'penguin':
            mesh.apply_transform(tm.transformations.rotation_matrix(np.radians(90), [1, 0, 0]))
        
    return mesh


mesh = load_and_preprocess_mesh(FILEPATH)

print(
    f"{MODEL}: {len(mesh.vertices)} verts, {len(mesh.faces)} faces, "
    f"watertight={mesh.is_watertight}, bodies={mesh.body_count}, "
    f"diagonal={mesh.scale:.3f}"
)

spot: 2930 verts, 5856 faces, watertight=True, bodies=1, diagonal=1.000


## Descriptor Function

There are many possible local feature descriptors that can be used to extract regions of the mesh. In this project, you will implement the shape diameter function as presented in the lecture.\
The function should take a mesh as input and return an array of shape diameter values, one value for each vertex. You may want to pass other parameters to the function, such as the number of rays used to sample the value of each vertex.\
Note that different meshes may require different parameter settings. Use the match statement to define the parameters for each mesh and use these variables in your function call:
```python
diam_values = compute_shape_diameter_function(mesh, num_rays, important_parameter=IMPORTANT_PARAMETER) # Define IMPORTANT_PARAMETER in the match statement
```

Because the shape diameter function returns a value for each vertex, but region growing is performed on the faces, you will also need to compute a per-face descriptor based on your vertex descriptors.\
There are multiple ways to do this. Try out different ones to see which one works best for the region growing task.

*Hint: trimesh provides a function to compute ray mesh intersections. If you additionally pip install pyembree/embreex, you can speed up the ray casting significantly!*\
*Hint: You may want to experiment with the log-alpha-normalization for a better region growing!*

**For your submission: use k3d to visualize both your vertex and face descriptors!**

In [399]:
def compute_shape_diameter_function(mesh, num_rays=30, cone_angle=60.0, eps=1e-4, seed=0):
    rng = np.random.default_rng(seed)
    V = mesh.vertices
    N = mesh.vertex_normals
    nv = len(V)

    # rays go inward (opposite to the outward normal)
    axis = -N
    half = np.deg2rad(cone_angle)

    # build a local coordinate frame for each vertex
    helper = np.tile([1.0, 0.0, 0.0], (nv, 1))
    helper[np.abs(axis[:, 0]) > 0.9] = [0.0, 1.0, 0.0]
    t1 = np.cross(axis, helper)
    t1 = t1 / np.linalg.norm(t1, axis=1, keepdims=True)
    t2 = np.cross(axis, t1)

    # sample random directions inside the cone
    u1 = rng.random((nv, num_rays))
    u2 = rng.random((nv, num_rays))
    cos_t = 1.0 - u1 * (1.0 - np.cos(half))
    sin_t = np.sqrt(np.clip(1.0 - cos_t**2, 0.0, 1.0))
    phi = 2.0 * np.pi * u2

    dirs = (cos_t[..., None] * axis[:, None, :]
            + sin_t[..., None] * (np.cos(phi)[..., None] * t1[:, None, :]
                                  + np.sin(phi)[..., None] * t2[:, None, :]))

    # nudge origins slightly inside the surface
    origins = (V - eps * N)[:, None, :] + np.zeros_like(dirs)
    O = origins.reshape(-1, 3)
    D = dirs.reshape(-1, 3)

    # weight: rays closer to the cone axis get higher weight
    axis_repeated = axis[:, None, :].repeat(num_rays, 1).reshape(-1, 3)
    cos_ang = (axis_repeated * D).sum(axis=1)
    ang = np.arccos(np.clip(cos_ang, -1.0, 1.0))
    w = 1.0 / np.maximum(ang, 1e-3)

    # cast all rays at once
    loc, idx_ray, idx_tri = mesh.ray.intersects_location(O, D, multiple_hits=False)
    dist = np.linalg.norm(loc - O[idx_ray], axis=1)

    # remove hits where the hit face points the same way as the origin normal
    hit_normals = mesh.face_normals[idx_tri]
    origin_normals = N[idx_ray // num_rays]
    keep = (hit_normals * origin_normals).sum(axis=1) < 0

    Dmat = np.full(nv * num_rays, np.nan)
    Wmat = np.full(nv * num_rays, np.nan)
    Dmat[idx_ray[keep]] = dist[keep]
    Wmat[idx_ray[keep]] = w[idx_ray[keep]]
    Dmat = Dmat.reshape(nv, num_rays)
    Wmat = Wmat.reshape(nv, num_rays)

    # remove outlier distances (more than 1 std from median)
    med = np.nanmedian(Dmat, axis=1, keepdims=True)
    sig = np.nanstd(Dmat, axis=1, keepdims=True)
    inlier = np.abs(Dmat - med) <= sig
    Wm = np.where(inlier, Wmat, np.nan)
    Dm = np.where(inlier, Dmat, np.nan)

    # weighted average
    f = np.nansum(Wm * Dm, axis=1) / np.nansum(Wm, axis=1)

    # fill vertices with no valid ray with the median
    bad = ~np.isfinite(f)
    f[bad] = np.nanmedian(f[~bad])

    return f


def log_alpha_normalize(values, alpha=4.0):
    # normalize to [0, 1] first
    v = (values - values.min()) / (values.max() - values.min())
    return np.log(v * alpha + 1.0) / np.log(alpha + 1.0)


In [400]:
from k3d.colormaps import matplotlib_color_maps as cmaps

def show_vertex_scalar(mesh, values, color_map=cmaps.Viridis):
    plot = k3d.plot()
    plot += k3d.mesh(
        mesh.vertices.astype('float32'),
        mesh.faces.astype('uint32'),
        attribute=values.astype('float32'),
        color_map=color_map,
        color_range=[float(values.min()), float(values.max())]
    )
    return plot

diam_values = compute_shape_diameter_function(mesh, num_rays=NUM_RAYS, cone_angle=CONE_ANGLE)
vertex_descriptor = log_alpha_normalize(diam_values, alpha=SDF_ALPHA)
show_vertex_scalar(mesh, vertex_descriptor).display()

C:\Users\Startklar\AppData\Local\Temp\ipykernel_45080\3342971172.py:57: RuntimeWarning: All-NaN slice encountered
  med = np.nanmedian(Dmat, axis=1, keepdims=True)
C:\Users\Startklar\AppData\Local\Temp\ipykernel_45080\3342971172.py:64: RuntimeWarning: invalid value encountered in divide
  f = np.nansum(Wm * Dm, axis=1) / np.nansum(Wm, axis=1)


Output()

In [401]:
def vertex_descriptor_to_face_descriptor(mesh, vertex_values, mode='mean'):
    # get the 3 vertex values for each face
    tri = vertex_values[mesh.faces]  # shape: (num_faces, 3)

    if mode == 'mean':
        return tri.mean(axis=1)
    if mode == 'min':
        return tri.min(axis=1)
    if mode == 'median':
        return np.median(tri, axis=1)

    raise ValueError(mode)

face_descriptor = vertex_descriptor_to_face_descriptor(mesh, vertex_descriptor, mode='mean')

In [402]:
def show_face_scalar(mesh, face_values, color_map=cmaps.Viridis):
    face_values = np.asarray(face_values, dtype=np.float32)

    print("num faces:", len(mesh.faces))
    print("num face values:", len(face_values))

    assert len(face_values) == len(mesh.faces), "face_values must have one value per face"

    # each face needs its own 3 vertices so we can color per face
    v = mesh.vertices[mesh.faces].reshape(-1, 3)
    f = np.arange(len(v)).reshape(-1, 3)

    # each vertex in a face gets the face's value
    attr = np.repeat(face_values, 3)

    print("exploded vertices:", v.shape)
    print("exploded faces:", f.shape)
    print("attribute shape:", attr.shape)

    lo = float(np.min(attr))
    hi = float(np.max(attr))

    if lo == hi:
        hi = lo + 1e-6

    plot = k3d.plot(camera_auto_fit=True)
    plot += k3d.mesh(
        np.ascontiguousarray(v, dtype=np.float32),
        np.ascontiguousarray(f, dtype=np.uint32),
        attribute=np.ascontiguousarray(attr, dtype=np.float32),
        color_map=color_map,
        color_range=[lo, hi],
        flat_shading=True,
        side='double'
    )

    return plot

show_face_scalar(mesh, face_descriptor).display()

num faces: 5856
num face values: 5856
exploded vertices: (17568, 3)
exploded faces: (5856, 3)
attribute shape: (17568,)


Output()

## Region Growing

Implement the region growing algorithm as presented in the lecture. The function should take a mesh and an array of per-face descriptors as input and return a list of clusters, where each cluster is a list of face indices. You may want to pass other parameters to the function, such as thresholds required for growing. Start growing from seed faces with local maximum features. Think of reasonable heuristics to decide when to stop growing a cluster and which faces to add.\
Since region growing may return a large number of clusters, you should also implement a function that merges clusters into neighboring clusters, if they are similar enough.\
Also here, think of reasonable heuristics to decide when to merge adjacent clusters.

*Hint: For the merging step, you can look at the volume of the convex hull of the individual clusters versus the possibly combined clusters to decide whether to merge them.*\
*Hint: trimesh provides several useful functions to compute properties of meshes (e.g. convex hull or volume). You can apply them to submeshes as well.*\
*Hint: Carefully read the documentation if you want to use trimesh functions. They may require properties your submeshes do not have.*

**For your submission: use k3d to visualize the clusters before and after merging!**

In [403]:
import heapq

def grow_regions(mesh, face_features, similarity_threshold=0.05):
    num_faces = len(mesh.faces)

    # Build adjacency list: for each face, which faces share an edge?
    face_adj = [[] for _ in range(num_faces)]
    for f0, f1 in mesh.face_adjacency:
        face_adj[f0].append(f1)
        face_adj[f1].append(f0)

    # Seeds = local maxima: faces whose descriptor >= all neighboring faces.
    # thickest points in their local neighborhood
    # Sort highest-first so we start with the most prominent features.
    seeds = [
        i for i in range(num_faces)
        if face_adj[i] and all(face_features[i] >= face_features[n] for n in face_adj[i])
    ]
    seeds.sort(key=lambda i: -face_features[i])

    assigned = np.full(num_faces, -1, dtype=int)
    clusters = []

    # loop until all elements are clustered
    for seed in seeds:
        if assigned[seed] != -1:
            continue  # already belongs to another cluster

        seed_val = face_features[seed]
        cluster_id = len(clusters)
        cluster = []

        # Min-heap: (difference from seed value, face index).
        # Faces most similar to the seed are processed first.
        heap = [(0.0, seed)]

        while heap:
            diff, face = heapq.heappop(heap)

            if assigned[face] != -1:
                continue
            if diff > similarity_threshold:
                # Every remaining heap entry will be at least as different, so stop.
                continue

            assigned[face] = cluster_id
            cluster.append(face)

            for nb in face_adj[face]:
                if assigned[nb] == -1:
                    nb_diff = abs(face_features[nb] - seed_val)
                    heapq.heappush(heap, (nb_diff, nb))

        clusters.append(cluster)

    # any face not reachable from a local maximum seed becomes its own cluster, will be absorbed by merge_regions
    for i in range(num_faces):
        if assigned[i] == -1:
            clusters.append([i])
            assigned[i] = len(clusters) - 1

    print(f"Region growing produced {len(clusters)} clusters.")
    return clusters


clusters = grow_regions(
    mesh, face_descriptor,
    similarity_threshold=SIMILARITY_THRESHOLD,
)

Region growing produced 550 clusters.


In [404]:
## Visualize initial regions
def show_clusters(mesh, clusters, title=""):
    rng = np.random.default_rng(42)
    n = len(clusters)

    # Random RGB color per cluster, packed as 0xRRGGBB uint32
    rgb = rng.integers(50, 220, size=(n, 3), dtype=np.uint32)
    colors_hex = (rgb[:, 0] << 16) | (rgb[:, 1] << 8) | rgb[:, 2]

    face_color = np.zeros(len(mesh.faces), dtype=np.uint32)
    for i, cluster in enumerate(clusters):
        face_color[np.array(cluster, dtype=int)] = colors_hex[i]

    # Explode mesh so every face has its own 3 vertices (needed for per-face color)
    verts = np.ascontiguousarray(mesh.vertices[mesh.faces].reshape(-1, 3), dtype=np.float32)
    faces = np.arange(len(verts), dtype=np.uint32).reshape(-1, 3)
    vert_colors = np.repeat(face_color, 3)  # same color for all 3 vertices of a face

    if title:
        print(title)
    print(f"  {n} clusters, {len(mesh.faces)} faces")

    plot = k3d.plot()
    plot += k3d.mesh(
        np.ascontiguousarray(verts),
        np.ascontiguousarray(faces),
        colors=np.ascontiguousarray(vert_colors),
        flat_shading=True,
        side='double',
    )
    return plot


show_clusters(mesh, clusters, title="Initial regions (before merging)").display()

Initial regions (before merging)
  550 clusters, 5856 faces


Output()

In [415]:
def merge_regions_new(
    mesh,
    clusters,
    face_features,
    convex_ratio,
    min_size,
    similarity_threshold,
):
    """
    Greedily merges adjacent mesh face clusters using a hybrid approach that balances 
    feature descriptor similarity and geometric convexity.
    
    Parameters:
    - mesh: trimesh object containing face and vertex data.
    - clusters: List of lists, where each sublist contains face indices belonging to a segment.
    - face_features: Array of features (e.g., normals, color, descriptor values) per face.
    - convex_ratio: Max allowed volume increase ratio when merging two convex hulls.
    - min_size: Face count threshold under which a cluster is considered a tiny artifact.
    - similarity_threshold: Maximum feature distance allowed for standard cluster merging.
    """

    # Convert all clusters to standard lists of indices to allow easy concatenation/manipulation
    clusters = [list(c) for c in clusters]

    def cluster_mean_feature(cluster):
        """Calculates the average feature descriptor value for all faces in a cluster."""
        return float(np.mean(face_features[np.array(cluster, dtype=int)]))

    def get_verts(face_list):
        """Extracts the unique 3D vertex coordinates associated with a list of face indices."""
        idx = np.unique(mesh.faces[np.array(face_list, dtype=int)].flatten())
        return mesh.vertices[idx]

    def hull_vol(verts):
        """
        Computes the volume of the convex hull around a set of 3D vertices.
        Returns a tiny fallback volume if the calculation fails or points are coplanar.
        """
        try:
            # A 3D convex hull requires at least 4 non-coplanar points. 
            # If a cluster is too small, skip calculation to avoid a Trimesh/Qhull crash.
            if len(verts) < 4:
                return 1e-10
            return tm.PointCloud(verts).convex_hull.volume
        except Exception:
            # Fallback for degenerate geometries (e.g., completely flat or linear point clusters)
            return 1e-10

    def find_adj_pairs(clusters):
        """Scans the mesh structure to find all adjacent pairs of different clusters."""
        # Create a lookup map: face_index -> cluster_index
        face_to_cluster = np.full(len(mesh.faces), -1, dtype=int)
        for i, c in enumerate(clusters):
            face_to_cluster[np.array(c, dtype=int)] = i

        pairs = set()
        # Check every internal shared edge across the entire mesh topology
        for f0, f1 in mesh.face_adjacency:
            c0 = face_to_cluster[f0]
            c1 = face_to_cluster[f1]

            # If both faces belong to valid, distinct clusters, they are neighbors
            if c0 >= 0 and c1 >= 0 and c0 != c1:
                # Store as an ordered tuple (min, max) to automatically eliminate duplicates in the set
                pairs.add((min(c0, c1), max(c0, c1)))

        return pairs

    # Flag to track whether any merges happened during a full iteration pass
    changed = True

    # Main iterative optimization loop
    while changed:
        changed = False

        # Find all currently adjacent cluster boundaries
        pairs = list(find_adj_pairs(clusters))

        # If no clusters are touching each other, we are done
        if not pairs:
            break

        merge_candidates = []

        # Step 1: Evaluate every adjacent pair and calculate merge eligibility + quality score
        for c0, c1 in pairs:
            # Compute descriptor distance between the two clusters
            feat0 = cluster_mean_feature(clusters[c0])
            feat1 = cluster_mean_feature(clusters[c1])
            feat_diff = abs(feat0 - feat1)

            # Retrieve 3D vertices for both individual clusters
            v0 = get_verts(clusters[c0])
            v1 = get_verts(clusters[c1])

            # Calculate individual convex hull volumes vs combined convex hull volume
            vol0 = hull_vol(v0)
            vol1 = hull_vol(v1)
            vol_combined = hull_vol(np.vstack([v0, v1]))

            # Ratio = combined_volume / sum_of_individual_volumes
            # Closer to 1.0 means the merge creates a highly natural, convex shape.
            ratio = vol_combined / max(vol0 + vol1, 1e-10)

            # Track cluster sizes to spot tiny "sliver" fragments/noise
            size0 = len(clusters[c0])
            size1 = len(clusters[c1])

            # Boolean heuristic flags
            small_pair = size0 < min_size or size1 < min_size
            similar_descriptor = feat_diff < similarity_threshold
            good_convex_merge = ratio < convex_ratio

            # Evaluate if this pair qualifies for a merge based on 3 distinct pathways:
            if (
                similar_descriptor                       # Condition A: They share highly similar features
                or good_convex_merge                     # Condition B: They fit perfectly together geometrically
                or (small_pair and feat_diff < 2.0 * similarity_threshold) # Condition C: One is a tiny artifact, so allow double feature tolerance to absorb it safely
            ):
                # Calculate a global "goodness" score. Lower scores mean a better match.
                # Weighting multiplier (0.2) balances feature importance against geometric volume ratio.
                score = feat_diff + 0.2 * ratio
                merge_candidates.append((score, c0, c1))

        # If no adjacent pairs met the threshold criteria, processing is complete
        if not merge_candidates:
            break

        # Step 2: Global Best-First Sorting.
        # Python sorts tuples by their first element by default (the quality score).
        merge_candidates.sort()

        used = set()           # Tracks which clusters have already committed to a merge in this pass
        merged_indices = set() # Tracks indices that have been absorbed and shouldn't be duplicated
        new_clusters = []

        # Step 3: Execute optimal merges while preventing a cluster from merging twice in one loop
        for _, c0, c1 in merge_candidates:
            # If either cluster has already merged with a better candidate earlier in this pass, skip
            if c0 in used or c1 in used:
                continue

            # Concatenate the face index lists together
            new_clusters.append(clusters[c0] + clusters[c1])

            # Mark both original cluster IDs as occupied
            used.add(c0)
            used.add(c1)
            merged_indices.add(c0)
            merged_indices.add(c1)

            # Toggle flag to guarantee the while loop runs another pass on the newly updated state
            changed = True

        # Step 4: Retain all clusters that did not undergo any merging during this pass
        for i, c in enumerate(clusters):
            if i not in merged_indices:
                new_clusters.append(c)

        # Overwrite the old cluster registry with the newly aggregated boundaries
        clusters = new_clusters

    print(f"After merging: {len(clusters)} clusters.")
    return clusters 


merged_clusters_new = merge_regions_new(
    mesh,
    clusters,
    face_descriptor,
    convex_ratio=CONVEX_RATIO,
    min_size=MERGE_MIN_SIZE,
    similarity_threshold = MERGE_FEATURE_THRESHOLD,
)

After merging: 18 clusters.


In [416]:
## Visualize merged regions
show_clusters(mesh, merged_clusters_new, title="Regions after merging").display()

Regions after merging
  18 clusters, 5856 faces


Output()

In [407]:


def merge_regions(mesh, clusters, convex_ratio, min_size):  # def merge_regions(mesh, clusters, convex_ratio=1.3, min_size=30):

    """
    Merge adjacent clusters using two heuristics (hint from instructions):

    1. Small clusters (< min_size faces) are always merged into a neighbor.
       We track the *current* merged size via union-find to avoid cascading.
    2. Two clusters are merged if their combined convex hull volume is not much
       larger than the sum of their individual convex hull volumes.
       Ratio = vol(hull(A ∪ B)) / (vol(hull(A)) + vol(hull(B)))
       If ratio < convex_ratio, the clusters likely belong to the same primitive.
    """

    clusters = [list(c) for c in clusters]

    def get_verts(face_list):
        idx = np.unique(mesh.faces[np.array(face_list, dtype=int)].flatten())
        return mesh.vertices[idx]

    def hull_vol(verts):
        try:
            return tm.PointCloud(verts).convex_hull.volume
        except Exception:
            return 1e-10

    def find_adj_pairs(clusters):
        face_to_cluster = np.full(len(mesh.faces), -1, dtype=int)
        for i, c in enumerate(clusters):
            face_to_cluster[np.array(c, dtype=int)] = i
        pairs = set()
        for f0, f1 in mesh.face_adjacency:
            c0, c1 = face_to_cluster[f0], face_to_cluster[f1]
            if c0 >= 0 and c1 >= 0 and c0 != c1:
                pairs.add((min(c0, c1), max(c0, c1)))
        return pairs

    changed = True
    while changed:
        changed = False
        n = len(clusters)
        parent = list(range(n))
        # Track current merged size so 'small' check doesn't cascade
        merged_size = [len(c) for c in clusters]

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def union(x, y):
            px, py = find(x), find(y)
            if px == py:
                return
            # Accumulate size into the new root
            merged_size[py] += merged_size[px]
            parent[px] = py

        for c0, c1 in find_adj_pairs(clusters):
            r0, r1 = find(c0), find(c1)
            if r0 == r1:
                continue

            # Use current merged sizes, not original sizes
            small = merged_size[r0] < min_size or merged_size[r1] < min_size

            if small:
                do_merge = True
            else:
                v0 = get_verts(clusters[c0])
                v1 = get_verts(clusters[c1])
                vol0 = hull_vol(v0)
                vol1 = hull_vol(v1)
                vol_combined = hull_vol(np.vstack([v0, v1]))
                sum_vol = vol0 + vol1
                do_merge = sum_vol > 0 and vol_combined / sum_vol < convex_ratio

            if do_merge:
                union(c0, c1)
                changed = True

        if changed:
            groups = {}
            for i in range(n):
                root = find(i)
                groups.setdefault(root, []).extend(clusters[i])
            clusters = list(groups.values())

    print(f"After merging: {len(clusters)} clusters.")
    return clusters


merged_clusters = merge_regions(
    mesh, clusters,
    convex_ratio=CONVEX_RATIO,
    min_size=MERGE_MIN_SIZE,
)


After merging: 21 clusters.


In [408]:
## Visualize merged regions
show_clusters(mesh, merged_clusters, title="Regions after merging").display()

Regions after merging
  21 clusters, 5856 faces


Output()

## Cylinder Fitting

Now that you have a set of clusters, you can fit a cylinder to each cluster. Implement the cylinder fitting algorithm as presented in the lecture. **Do not use any other primitives!**\
The function `fit_cylinder` should take a (sub)mesh as input and return suitable parameters `(centroid, axis, height, radius)` for a cylinder.\
Similarly, the function `fit_cylinders` should take a mesh and a set of clusters as input and return a list of cylinder parameters, one for each cluster.

*Hint: For the shape operator, you may want to consider a larger neighborhood per vertex (with suitable weighting) for robust results.*\
*Hint: You can use the functions provided in the top of the notebook for your cylinder fitting and visualization.*

**For your submission: use k3d to visualize the fitted cylinders!**

In [ ]:


def fit_cylinder(points, normals, sigma=None):

    points = np.asarray(points, dtype=float)
    normals = np.asarray(normals, dtype=float)
    num_points = points.shape[0]
    
    
    #  Extended Shape Operator to find Minimal Principal Curvature Axis
   
    # Automatically estimate sigma if not provided (10% of maximum cluster width)
    if sigma == None:
        spatial_span = np.ptp(points, axis=0).max()
        sigma = spatial_span * 0.1 if spatial_span > 0 else 1.0

    M_global = np.zeros((3, 3))
    
    # accumulate the Shape Operator matrix across all points in the cluster
    for i in range(num_points):
        p_coord = points[i]
        p_normal = normals[i]
        
        # normals must be unit vectors
        norm_len = np.linalg.norm(p_normal)
        if norm_len > 1e-8:
            p_normal = p_normal / norm_len
            
        # exclude the current point itself from its neighborhood pool
        mask = np.ones(num_points, dtype=bool)
        mask[i] = False
        neighbors = points[mask]
        
        # displacement vectors and distances
        diff = neighbors - p_coord
        dist_sq = np.sum(diff**2, axis=1)
        dist = np.sqrt(dist_sq)
        
        # protect against overlapping duplicate points                                        
        valid = dist > 1e-8
        if not np.any(valid):
            continue
        diff, dist, dist_sq = diff[valid], dist[valid], dist_sq[valid]
        
        # 2. tangent projection vectors (T_j)
        dot_p_diff = diff @ p_normal
        proj_diff = diff - dot_p_diff[:, None] * p_normal
        proj_norm = np.linalg.norm(proj_diff, axis=1, keepdims=True)
        proj_norm[proj_norm == 0] = 1.0
        T = proj_diff / proj_norm
        
        # 3. weights (w_j) and Directional Curvatures (k_j)
        w = np.exp(-dist / sigma)
        k = (2.0 * dot_p_diff) / dist_sq
        
        # 4. construct local Mp matrix and add to global sum
        coefficients = w * k
        M_global += T.T @ (T * coefficients[:, None])
        
    # Standardize the aggregated matrix by the size of the cluster
    M_global /= num_points
    
    # Eigendecomposition of the global shape operator matrix
    evals, evecs = np.linalg.eigh(M_global)
    
    # smallest eigenvalue corresponds to the direction of minimal curvature 
    axis = evecs[:, 0]
    axis = axis / np.linalg.norm(axis)  
    
 
    # project the points of the cluster onto a 2D plane
  
    if np.abs(axis[0]) > np.abs(axis[1]):
        u = np.array([-axis[2], 0.0, axis[0]])
    else:
        u = np.array([0.0, axis[2], -axis[1]])
    u = u / np.linalg.norm(u)
    v = np.cross(axis, u)
    
    x_2d = points @ u
    y_2d = points @ v
    
    # in this 2D plane, find the center and radius of the circle
   
    A_mat = np.stack([2.0 * x_2d, 2.0 * y_2d, np.ones_like(x_2d)], axis=1)
    b_vec = x_2d**2 + y_2d**2
    
    w_vec, _, _, _ = np.linalg.lstsq(A_mat, b_vec, rcond=None)
    x_c, y_c, const = w_vec[0], w_vec[1], w_vec[2]
    
    r_squared = const + x_c**2 + y_c**2
    if r_squared <= 0:
        raise ValueError("Degenerate point layout: cannot compute a physical radius.")
    radius = np.sqrt(r_squared)
    
    center_3d_on_plane = x_c * u + y_c * v
    
    
    # cut the cylinder based on maximum extended points 
    heights = points @ axis
    h_min = np.min(heights)
    h_max = np.max(heights)
    height = h_max - h_min
    
    centroid = center_3d_on_plane + ((h_min + h_max) / 2.0) * axis
    
    return centroid, axis, height, radius 




In [410]:
## def fit_cylinders(mesh, clusters, ...)

def fit_cylinders(mesh, clusters, sigma=None):

    fitted_tuples = []
    
    for i, face_indices in enumerate(clusters):
        vertex_indices = np.unique(mesh.faces[face_indices])
        pts = np.asarray(mesh.vertices[vertex_indices], dtype=float)
        nrms = np.asarray(mesh.vertex_normals[vertex_indices], dtype=float)
        
        if pts.shape[0] < 3:
            continue
            
        # downsample for Extended Shape Operator speed if the cluster is huge
        if pts.shape[0] > 800:
            step = pts.shape[0] // 400
            fit_pts = pts[::step]
            fit_nrms = nrms[::step]
        else:
            fit_pts = pts
            fit_nrms = nrms
            
        try:
            centroid, axis, height, radius = fit_cylinder(fit_pts, fit_nrms, sigma=sigma)
            
            fitted_tuples.append((centroid, axis, height, radius))
            print(f"Cluster {i} parsed -> Radius: {radius:.3f}, Height: {height:.3f}")
        except Exception as e:
            print(f"Skipping cluster {i} due to calculation error: {e}")
            
    return fitted_tuples

In [411]:
def get_cylinder_mesh(centroid, axis, height, radius, resolution=32, color=0xFF4500):
    
    # generate canonical cylinder vertices around the Z-axis
    theta = np.linspace(0, 2 * np.pi, resolution, endpoint=False)
    x = radius * np.cos(theta)
    y = radius * np.sin(theta)
    
    v_bottom = np.stack([x, y, np.full_like(x, -height / 2.0)], axis=1)
    v_top = np.stack([x, y, np.full_like(x, height / 2.0)], axis=1)
    
    # center points for the bottom and top circular caps
    v_bottom_center = np.array([[0.0, 0.0, -height / 2.0]])
    v_top_center = np.array([[0.0, 0.0, height / 2.0]])
    
    # stack everything together into one array
    vertices = np.vstack([v_bottom, v_top, v_bottom_center, v_top_center])
    
    # re-orient vertices to point along the target tracking axis
    axis = axis / np.linalg.norm(axis)
    z_axis = np.array([0.0, 0.0, 1.0])
    
    if not np.allclose(axis, z_axis):
        if np.allclose(axis, -z_axis):
            vertices[:, 2] *= -1.0
        else:
            v = np.cross(z_axis, axis)
            s = np.linalg.norm(v)
            c = np.dot(z_axis, axis)
            v_skew = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
            R = np.eye(3) + v_skew + (v_skew @ v_skew) * ((1.0 - c) / (s ** 2))
            vertices = vertices @ R.T
            
    # move to the physical center position
    vertices += centroid
    
    # solid triangular faces (Walls + Closed Caps)
    faces = []
    idx_b_center = 2 * resolution      # Index of bottom cap center
    idx_t_center = 2 * resolution + 1  # Index of top cap center
    
    for i in range(resolution):
        next_i = (i + 1) % resolution
        
        # walls 
        faces.append([i, next_i, i + resolution])
        faces.append([i + resolution, next_i, next_i + resolution])
        # bottom cap 
        faces.append([idx_b_center, next_i, i])
        # top cap
        faces.append([idx_t_center, i + resolution, next_i + resolution])
        
    faces = np.array(faces, dtype=np.uint32)
    
    
    return k3d.mesh(
        vertices=vertices.astype(np.float32),
        indices=faces.astype(np.uint32),
        color=color,
        opacity=1.0,  
        side='double'
    )


# Visualize
cylinders = fit_cylinders(mesh, merged_clusters)

plot = k3d.plot()

#draw cylinders
for (centroid, axis, height, radius) in cylinders:
    cylinder_mesh = get_cylinder_mesh(centroid, axis, height, radius)
    plot += cylinder_mesh

plot.display()


Cluster 0 parsed -> Radius: 0.058, Height: 0.100
Cluster 1 parsed -> Radius: 0.172, Height: 0.300
Cluster 2 parsed -> Radius: 0.111, Height: 0.203
Cluster 3 parsed -> Radius: 0.023, Height: 0.025
Cluster 4 parsed -> Radius: 0.065, Height: 0.194
Cluster 5 parsed -> Radius: 0.052, Height: 0.223
Cluster 6 parsed -> Radius: 0.049, Height: 0.142
Cluster 7 parsed -> Radius: 0.134, Height: 0.282
Cluster 8 parsed -> Radius: 0.024, Height: 0.067
Cluster 9 parsed -> Radius: 0.046, Height: 0.199
Cluster 10 parsed -> Radius: 0.203, Height: 0.267
Cluster 11 parsed -> Radius: 0.047, Height: 0.074
Cluster 12 parsed -> Radius: 0.090, Height: 0.196
Cluster 13 parsed -> Radius: 0.094, Height: 0.227
Cluster 14 parsed -> Radius: 0.063, Height: 0.133
Cluster 15 parsed -> Radius: 0.061, Height: 0.137
Cluster 16 parsed -> Radius: 0.039, Height: 0.103
Cluster 17 parsed -> Radius: 0.059, Height: 0.137
Cluster 18 parsed -> Radius: 0.064, Height: 0.146
Cluster 19 parsed -> Radius: 0.038, Height: 0.107
Cluster 20

Output()